# Tsetlin bake-off on a free Colab GPU

`tmu` is only fast with a CUDA GPU. On this machine it hangs; here it flies.

**Runtime → Change runtime type → T4 GPU** before running.

This clones the repo, builds features (synthetic until real ones are committed),
trains a weighted Tsetlin Machine on the same time-split as the bake-off, and
prints its score next to the baselines plus the clauses it learned.

In [ ]:
!nvidia-smi -L || echo 'NO GPU - set Runtime > Change runtime type > T4 GPU'
!git clone --depth 1 https://github.com/naibwedi/tsetlin-market-lab.git
%cd tsetlin-market-lab
!pip -q install 'numpy<2' 'scikit-learn==1.5.2' pandas pyarrow pyyaml python-dotenv xgboost lightgbm tmu pycuda

GPU 0: Tesla T4 (UUID: GPU-1f2bdea0-c9a3-0844-a509-497294be2301)
Cloning into 'tsetlin-market-lab'...
remote: Enumerating objects: 54, done.
remote: Counting objects: 100% (54/54), done.
remote: Compressing objects: 100% (48/48), done.
remote: Total 54 (delta 1), reused 39 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (54/54), 52.31 KiB | 13.08 MiB/s, done.
Resolving deltas: 100% (1/1), done.
/content/tsetlin-market-lab
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.8/15.8 MB 67.4 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Installing backend dependencies ... done


In [ ]:
import os, glob
if not glob.glob('data/features/X.parquet'):
    !python -m src.ingest.make_synthetic --n-matches 80
    !python -m src.panel.build_panel --config config/features.yaml
    !python -m src.features.booleanize --config config/features.yaml
print('features ready:', glob.glob('data/features/*'))

In [ ]:
# Baselines (fast) for the reference leaderboard
!python -m src.models.bakeoff --config config/bakeoff.ci.yaml
print(open('results/summary.md').read())

In [ ]:
import json, numpy as np, time
from sklearn.metrics import roc_auc_score, average_precision_score
from tmu.models.classification.vanilla_classifier import TMClassifier
from src.common.config import load_yaml
from src.models.bakeoff import load_split, score

sp = load_split(load_yaml('config/bakeoff.yaml'))
Xtr = sp.X[sp.tr | sp.va].astype(np.uint32); ytr = sp.y[sp.tr | sp.va].astype(np.uint32)
Xte = sp.X[sp.te].astype(np.uint32); yte = sp.y[sp.te].astype(int)
print('train', Xtr.shape, 'test', Xte.shape, 'pos rate', round(yte.mean(), 3))

tm = TMClassifier(number_of_clauses=2000, T=32, s=5.0, weighted_clauses=True, platform='CUDA', seed=0)
t0 = time.time()
for epoch in range(60):
    tm.fit(Xtr, ytr)
    if epoch % 10 == 9:
        _, cs = tm.predict(Xte, return_class_sums=True)
        cs = np.asarray(cs, float); proba = 1/(1+np.exp(-(cs[:,1]-cs[:,0])/32))
        print(f'epoch {epoch+1:2d}  AUC={roc_auc_score(yte, proba):.3f}  ({time.time()-t0:.0f}s)')

m = score(sp.y[sp.te], proba, 0.1)
print('\nTsetlin Machine:', json.dumps({k: round(v,4) for k,v in m.items()}))

In [ ]:
# Learned clauses
n_lit = len(sp.feat); shown = 0
for cls in (1, 0):
    print('\n=== predicts', 'MOVE' if cls else 'NO-MOVE', '===')
    for c in range(tm.number_of_clauses):
        lits = [sp.feat[k] for k in range(n_lit) if tm.get_ta_action(clause=c, ta=k, the_class=cls)]
        lits += ['NOT '+sp.feat[k] for k in range(n_lit) if tm.get_ta_action(clause=c, ta=k+n_lit, the_class=cls)]
        if lits and 0 < len(lits) <= 5 and shown < 20:
            print('IF', ' AND '.join(lits)); shown += 1
    shown = 0